# TIGER vs Content Retrieval — 重叠与分桶分析

In [27]:
import numpy as np
import pandas as pd
import json
from collections import defaultdict

# ── 1. 加载数据 ──

# 召回列表
tiger = np.load('../predictions/tiger_baseline_ce_top50.npy')          # (N, 50)
content = np.load('../predictions/content_sim_last1_top50.npy')            # (N, 50)

# 测试集
test = pd.read_parquet('../data/Beauty/test.parquet')
targets = test['target'].values
histories = test['history'].values
N = len(test)
print(f"样本数: {N}")
print(f"TIGER preds: {tiger.shape}, Content preds: {content.shape}")

样本数: 22363
TIGER preds: (22363, 50), Content preds: (22363, 50)


In [28]:
# 内容嵌入（用于相似度分桶）
emb_df = pd.read_parquet('../data/Beauty/item_emb.parquet')
max_id = int(emb_df['ItemID'].max())
emb = np.zeros((max_id + 1, 768), dtype=np.float32)
for row in emb_df.itertuples(index=False):
    emb[row.ItemID] = np.array(row.embedding, dtype=np.float32)
# L2 normalize
norms = np.linalg.norm(emb, axis=-1, keepdims=True)
norms = np.where(norms > 0, norms, 1.0)
emb_norm = emb / norms
print(f"Embeddings: {emb_norm.shape}")

Embeddings: (12102, 768)


In [29]:
# Train data（算 popularity）
train = pd.read_parquet('../data/Beauty/train.parquet')
pop_counter = defaultdict(int)
for hist, tgt in zip(train['history'], train['target']):
    for item in hist:
        pop_counter[item] += 1
    pop_counter[tgt] += 1
print(f"Train items with counts: {len(pop_counter)}")

# Metadata（算 category）
item_mapping = np.load('../data/Beauty/item_mapping.npy', allow_pickle=True).item()
id2idx = {asin: iid for asin, iid in item_mapping.items()}

# 解析 metadata JSON lines — 每个 item 可能有多个 category path
asin_to_cats = {}
with open('../data/Beauty/Beauty_metadata.json') as f:
    for line in f:
        obj = json.loads(line)
        asin = obj['asin']
        cats = obj.get('categories', [])
        if cats:
            asin_to_cats[asin] = [tuple(c) for c in cats if c]
print(f"Metadata items with categories: {len(asin_to_cats)}")

# item_id → list of category path tuples
item_to_cats = {}
for asin, iid in id2idx.items():
    if asin in asin_to_cats:
        item_to_cats[iid] = asin_to_cats[asin]
print(f"Dataset items with category mapping: {len(item_to_cats)}")
multi_cat_items = sum(1 for v in item_to_cats.values() if len(v) > 1)
print(f"  of which multi-path: {multi_cat_items}")

Train items with counts: 12068
Metadata items with categories: 259196
Dataset items with category mapping: 12101
  of which multi-path: 12


## 2. 重叠分析

### 2a. Target 命中重叠

对每个用户，哪路命中了 target？

In [30]:
def target_overlap_analysis(tiger_preds, content_preds, targets, ks=[5, 10, 20, 50]):
    results = []
    for k in ks:
        tiger_hit = np.any(tiger_preds[:, :k] == targets[:, None], axis=1)
        content_hit = np.any(content_preds[:, :k] == targets[:, None], axis=1)
        
        both = tiger_hit & content_hit
        tiger_only = tiger_hit & ~content_hit
        content_only = ~tiger_hit & content_hit
        
        results.append({
            'k': k,
            'TIGER Recall': tiger_hit.mean(),
            'Content Recall': content_hit.mean(),
            'Both hit': both.sum(),
            'TIGER only': tiger_only.sum(),
            'Content only': content_only.sum(),
            'Union Recall': (tiger_hit | content_hit).mean(),
        })
    return pd.DataFrame(results)

target_overlap_df = target_overlap_analysis(tiger, content, targets)
target_overlap_df

,k,TIGER Recall,Content Recall,Both hit,TIGER only,Content only,Union Recall
0,5,0.034611,0.043912,71,703,911,0.075348
1,10,0.056567,0.058892,170,1095,1147,0.107857
2,20,0.083531,0.077897,330,1538,1412,0.146671
3,50,0.137012,0.111076,702,2362,1782,0.216697


### 2b. Item 列表重叠

对每个用户，统计两路 top-k 召回列表的 item 重叠程度。

In [31]:
def item_overlap_analysis(tiger_preds, content_preds, ks=[5, 10, 20, 50]):
    results = []
    for k in ks:
        n = len(tiger_preds)
        total_intersection = 0
        total_tiger_only = 0
        total_content_only = 0
        total_union = 0
        for i in range(n):
            t_set = set(tiger_preds[i, :k]) - {0}
            c_set = set(content_preds[i, :k]) - {0}
            total_intersection += len(t_set & c_set)
            total_tiger_only += len(t_set - c_set)
            total_content_only += len(c_set - t_set)
            total_union += len(t_set | c_set)
        results.append({
            'k': k,
            'Intersection/user': total_intersection / n,
            'TIGER-only/user': total_tiger_only / n,
            'Content-only/user': total_content_only / n,
            'Jaccard': total_intersection / total_union if total_union > 0 else 0,
        })
    return pd.DataFrame(results)

item_overlap_df = item_overlap_analysis(tiger, content)
item_overlap_df

,k,Intersection/user,TIGER-only/user,Content-only/user,Jaccard
0,5,0.072888,4.927112,4.927112,0.007342
1,10,0.231275,9.768725,9.768725,0.011699
2,20,0.681796,19.318204,19.318204,0.017340
3,50,2.538255,47.446362,47.461745,0.026048


### 2c. Union Recall（RRF 合并）

In [32]:
def union_recall_rrf(tiger_preds, content_preds, targets, ks=[5, 10, 20, 50], rrf_k=60):
    max_k = max(ks)
    union_preds = np.zeros((len(targets), max_k), dtype=np.int32)
    
    for i in range(len(targets)):
        scores = defaultdict(float)
        for rank, item in enumerate(tiger_preds[i]):
            if item != 0:
                scores[item] += 1.0 / (rrf_k + rank + 1)
        for rank, item in enumerate(content_preds[i]):
            if item != 0:
                scores[item] += 1.0 / (rrf_k + rank + 1)
        
        merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:max_k]
        for j, (item, _) in enumerate(merged):
            union_preds[i, j] = item
    
    results = {}
    for k in ks:
        hit = np.any(union_preds[:, :k] == targets[:, None], axis=1)
        results[f'Union Recall@{k}'] = hit.mean()
    return results

rrf_results = union_recall_rrf(tiger, content, targets)
for name, val in rrf_results.items():
    print(f"{name}: {val:.4f}")

Union Recall@5: 0.0504
Union Recall@10: 0.0764
Union Recall@20: 0.1118
Union Recall@50: 0.1661


## 3. 分桶分析

In [33]:
# 通用分桶分析函数

def compute_metrics(preds, targets, bucket_mask, bucket_name, ks=[5, 10, 20]):
    p = preds[bucket_mask]
    t = targets[bucket_mask]
    n = len(t)
    if n == 0:
        return {'bucket': bucket_name, 'n': 0}
    
    metrics = {'bucket': bucket_name, 'n': n}
    for k in ks:
        hit = np.any(p[:, :k] == t[:, None], axis=1)
        recall = hit.mean()
        match_pos = np.argmax(p[:, :k] == t[:, None], axis=1)
        ndcg_vals = np.where(hit, 1.0 / np.log2(match_pos + 2.0), 0.0)
        ndcg = ndcg_vals.mean()
        metrics[f'Recall@{k}'] = recall
        metrics[f'NDCG@{k}'] = ndcg
    return metrics


def bucket_analysis(tiger_preds, content_preds, targets, masks, bucket_names, ks=[5, 10, 20]):
    rows = []
    for mask, name in zip(masks, bucket_names):
        t_metrics = compute_metrics(tiger_preds, targets, mask, name, ks)
        c_metrics = compute_metrics(content_preds, targets, mask, name, ks)
        n = t_metrics.get('n', 0)
        row = {'bucket': name, 'n': n}
        for k in ks:
            t_r = t_metrics.get(f'Recall@{k}', 0)
            c_r = c_metrics.get(f'Recall@{k}', 0)
            row[f'TIGER R@{k}'] = t_r
            row[f'TIGER N@{k}'] = t_metrics.get(f'NDCG@{k}', 0)
            row[f'Content R@{k}'] = c_r
            row[f'Content N@{k}'] = c_metrics.get(f'NDCG@{k}', 0)
            row[f'delta R@{k}'] = c_r - t_r
        rows.append(row)
    return pd.DataFrame(rows)

### 3a. 内容相似度分桶

target 与 last item 的 cosine similarity，按三分位数分为 high / medium / low。

In [ ]:
# 计算每个样本的 target-last_item cosine similarity
sims = []
for hist, tgt in zip(histories, targets):
    hist_list = list(hist)
    if len(hist_list) > 0 and tgt > 0 and hist_list[-1] > 0:
        tgt_vec = emb_norm[tgt]
        last_vec = emb_norm[hist_list[-1]]
        sims.append(np.dot(tgt_vec, last_vec))
    else:
        sims.append(np.nan)

sims = np.array(sims)
lo = np.nanpercentile(sims, 33.3)
hi = np.nanpercentile(sims, 66.7)
print(f"Similarity thresholds: low<{lo:.3f}, medium<{hi:.3f}, high>={hi:.3f}")

sim_masks = [sims <= lo, (sims > lo) & (sims < hi), sims >= hi]
sim_names = ['Low similarity', 'Medium similarity', 'High similarity']

sim_df = bucket_analysis(tiger, content, targets, sim_masks, sim_names)
sim_df

Similarity thresholds: low<0.860, medium<0.890, high>=0.890


,bucket,n,TIGER R@5,TIGER N@5,Content R@5,Content N@5,delta R@5,TIGER R@10,TIGER N@10,Content R@10,Content N@10,delta R@10,TIGER R@20,TIGER N@20,Content R@20,Content N@20,delta R@20
0,Low similarity,7447,0.014234,0.008933,0.000000,0.000000,-0.014234,0.022425,0.011561,0.00000,0.000000,-0.022425,0.036525,0.015112,0.000000,0.000000,-0.036525
1,Medium similarity,7469,0.025840,0.017604,0.000000,0.000000,-0.025840,0.044584,0.023701,0.00000,0.000000,-0.044584,0.066542,0.029216,0.000134,0.000033,-0.066408
2,High similarity,7447,0.063784,0.041606,0.131865,0.093899,0.068081,0.102726,0.054195,0.17685,0.108473,0.074124,0.147576,0.065552,0.233785,0.122875,0.086209


### 3b. 用户历史长度分桶

≤3 / 4-6 / >6。

In [39]:
hist_lens = np.array([len(list(h)) for h in histories])
print(f"History length: min={hist_lens.min()}, 25%={np.percentile(hist_lens,25):.0f}, median={np.median(hist_lens):.0f}, 75%={np.percentile(hist_lens,75):.0f}, max={hist_lens.max()}")

# 用三分位数动态分桶（实际分位点会落在整数上）
lo_len = int(np.percentile(hist_lens, 33.3))
hi_len = int(np.percentile(hist_lens, 66.7))
print(f"Thresholds: short<={lo_len}, medium<={hi_len}, long>={hi_len+1}")
print(f"  short (<={lo_len}): {(hist_lens<=lo_len).sum()}, medium ({lo_len+1}-{hi_len}): {((hist_lens>lo_len)&(hist_lens<=hi_len)).sum()}, long (>{hi_len}): {(hist_lens>hi_len).sum()}")

len_masks = [hist_lens <= lo_len, (hist_lens > lo_len) & (hist_lens <= hi_len), hist_lens > hi_len]
len_names = [f'Short (≤{lo_len})', f'Medium ({lo_len+1}-{hi_len})', f'Long (>{hi_len})']

len_df = bucket_analysis(tiger, content, targets, len_masks, len_names)
len_df

History length: min=4, 25%=4, median=5, 75%=8, max=203
Thresholds: short<=5, medium<=7, long>=8
  short (<=5): 11384, medium (6-7): 4490, long (>7): 6489


,bucket,n,TIGER R@5,TIGER N@5,Content R@5,Content N@5,delta R@5,TIGER R@10,TIGER N@10,Content R@10,Content N@10,delta R@10,TIGER R@20,TIGER N@20,Content R@20,Content N@20,delta R@20
0,Short (≤5),11384,0.033995,0.021919,0.044009,0.031318,0.010014,0.053408,0.028201,0.059821,0.036461,0.006413,0.079146,0.034697,0.080025,0.041570,0.000878
1,Medium (6-7),4490,0.033408,0.022142,0.043207,0.029617,0.009800,0.057906,0.030049,0.056793,0.034000,-0.001114,0.082628,0.036228,0.074165,0.038375,-0.008463
2,Long (>7),6489,0.036523,0.024489,0.044229,0.032327,0.007705,0.061180,0.032478,0.058715,0.036995,-0.002466,0.091848,0.040264,0.076745,0.041573,-0.015102


### 3c. Target Popularity 分桶

按 target 在训练集中出现次数的三分位数分为 tail / mid / head。

In [36]:
target_pops = np.array([pop_counter.get(t, 0) for t in targets])
pop_lo = np.percentile(target_pops, 33.3)
pop_hi = np.percentile(target_pops, 66.7)
print(f"Popularity thresholds: tail<{pop_lo:.0f}, mid<{pop_hi:.0f}, head>={pop_hi:.0f}")
print(f"  tail: {(target_pops<=pop_lo).sum()}, mid: {((target_pops>pop_lo)&(target_pops<pop_hi)).sum()}, head: {(target_pops>=pop_hi).sum()}")

pop_masks = [target_pops <= pop_lo, (target_pops > pop_lo) & (target_pops < pop_hi), target_pops >= pop_hi]
pop_names = ['Tail', 'Mid', 'Head']

pop_df = bucket_analysis(tiger, content, targets, pop_masks, pop_names)
pop_df

Popularity thresholds: tail<7, mid<22, head>=22
  tail: 8273, mid: 6466, head: 7624


,bucket,n,TIGER R@5,TIGER N@5,Content R@5,Content N@5,delta R@5,TIGER R@10,TIGER N@10,Content R@10,Content N@10,delta R@10,TIGER R@20,TIGER N@20,Content R@20,Content N@20,delta R@20
0,Tail,8273,0.001451,0.000733,0.053427,0.038262,0.051976,0.003264,0.001319,0.069382,0.043439,0.066119,0.006286,0.002085,0.089810,0.048612,0.083525
1,Mid,6466,0.014538,0.009610,0.044541,0.031159,0.030003,0.025982,0.013302,0.061862,0.036760,0.035880,0.043767,0.017774,0.080575,0.041492,0.036808
2,Head,7624,0.087618,0.057667,0.033054,0.023774,-0.054565,0.140346,0.074736,0.044990,0.027641,-0.095357,0.201076,0.090077,0.062697,0.032115,-0.138379


### 3d. 类目跳转分桶（L2 / L3）

last item 与 target 的 category 前缀匹配对比。L1 全是 Beauty 跳过。

In [37]:
def check_category_match(last_item, tgt, level):
    """Check if last_item and target share the same category prefix at given level (0-indexed).
    Returns 0=same, 1=different, -1=unknown."""
    last_paths = item_to_cats.get(last_item, [])
    tgt_paths = item_to_cats.get(tgt, [])
    if not last_paths or not tgt_paths:
        return -1
    last_prefixes = set(p[:level+1] for p in last_paths if len(p) > level)
    tgt_prefixes = set(p[:level+1] for p in tgt_paths if len(p) > level)
    if not last_prefixes or not tgt_prefixes:
        return -1
    return 0 if (last_prefixes & tgt_prefixes) else 1

for level in [1, 2]:  # L2, L3
    cat_labels = [check_category_match(list(h)[-1] if len(list(h)) > 0 else 0, t, level)
                  for h, t in zip(histories, targets)]
    cat_labels = np.array(cat_labels)
    print(f"\n--- L{level+1} category ---")
    print(f"Same: {(cat_labels==0).sum()}, Different: {(cat_labels==1).sum()}, Unknown: {(cat_labels==-1).sum()}")
    
    masks = [cat_labels == 0, cat_labels == 1]
    names = [f'L{level+1} same category', f'L{level+1} diff category']
    df = bucket_analysis(tiger, content, targets, masks, names)
    print(df.to_string(index=False))


--- L2 category ---
Same: 9792, Different: 12571, Unknown: 0
          bucket     n  TIGER R@5  TIGER N@5  Content R@5  Content N@5  delta R@5  TIGER R@10  TIGER N@10  Content R@10  Content N@10  delta R@10  TIGER R@20  TIGER N@20  Content R@20  Content N@20  delta R@20
L2 same category  9792   0.047488   0.030842     0.090380     0.064532   0.042892    0.077717    0.040610      0.119792      0.074066    0.042075    0.114685    0.049981      0.156761      0.083423    0.042075
L2 diff category 12571   0.024580   0.016375     0.007716     0.005359  -0.016864    0.040092    0.021403      0.011455      0.006566   -0.028637    0.059263    0.026212      0.016466      0.007829   -0.042797

--- L3 category ---
Same: 5079, Different: 16978, Unknown: 306
          bucket     n  TIGER R@5  TIGER N@5  Content R@5  Content N@5  delta R@5  TIGER R@10  TIGER N@10  Content R@10  Content N@10  delta R@10  TIGER R@20  TIGER N@20  Content R@20  Content N@20  delta R@20
L3 same category  5079   0.058279 

## 4. 汇总结论

In [40]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print("\n--- Target hit overlap ---")
print(target_overlap_df.to_string(index=False))

print("\n--- Item list overlap ---")
print(item_overlap_df.to_string(index=False))

print("\n--- Union with RRF ---")
for name, val in rrf_results.items():
    print(f"  {name}: {val:.4f}")

print("\n--- Similarity buckets ---")
print(sim_df.to_string(index=False))

print("\n--- History length buckets ---")
print(len_df.to_string(index=False))

print("\n--- Popularity buckets ---")
print(pop_df.to_string(index=False))

print("\n--- Category jump (L2/L3) --- see cell output above ---")

SUMMARY

--- Target hit overlap ---
 k  TIGER Recall  Content Recall  Both hit  TIGER only  Content only  Union Recall
 5      0.034611        0.043912        71         703           911      0.075348
10      0.056567        0.058892       170        1095          1147      0.107857
20      0.083531        0.077897       330        1538          1412      0.146671
50      0.137012        0.111076       702        2362          1782      0.216697

--- Item list overlap ---
 k  Intersection/user  TIGER-only/user  Content-only/user  Jaccard
 5           0.072888         4.927112           4.927112 0.007342
10           0.231275         9.768725           9.768725 0.011699
20           0.681796        19.318204          19.318204 0.017340
50           2.538255        47.446362          47.461745 0.026048

--- Union with RRF ---
  Union Recall@5: 0.0504
  Union Recall@10: 0.0764
  Union Recall@20: 0.1118
  Union Recall@50: 0.1661

--- Similarity buckets ---
           bucket    n  TIGER R@